# Deep Past Initiative — ByT5 (Participant 1: Byte-Level Seq2Seq)


## 1. Install Dependencies


In [1]:
!pip install -q -U transformers datasets accelerate peft sentencepiece evaluate sacrebleu mlflow huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 94.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 109.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 54.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━

## 2. Imports


In [2]:
import os
import sys
import glob
import gc
import json
import random

import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)

import evaluate
import mlflow
from huggingface_hub import login as hf_login

## 3. Configuration and Tokens

Replace the placeholder values with your own paths and secrets before running. Never commit a real `HF_TOKEN` to a public repository; use environment variables or Colab/Kaggle Secrets instead.


## 3.1. Experiment Tracking with MLflow + DagsHub


In [3]:
# ============================================================
# PLACEHOLDERS — replace with real values before running
# ============================================================
AUTHOR_NAME = "name"
HF_USERNAME = "userName"

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

os.environ["MLFLOW_TRACKING_USERNAME"] = user_secrets.get_secret("MLFLOW_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = user_secrets.get_secret("MLFLOW_PASSWORD")

MLFLOW_TRACKING_URI = user_secrets.get_secret("MLFLOW_TRACKING_URI")
MLFLOW_EXPERIMENT_NAME = "Byte-Level-Seq2Seq-machine-translation"

# Load competition data directly; no pipeline or normalization here
TRAIN_DATA_PATH = "/kaggle/input/competitions/deep-past-initiative-machine-translation/train.csv"
TEST_DATA_PATH  = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
OUTPUT_DIR = "outputs"
MODEL_ARCHITECTURE_TAG = "byt5"
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if HF_TOKEN:
    hf_login(token=HF_TOKEN)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


## 4. Load Dataset


In [4]:
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv(TRAIN_DATA_PATH)
df = df.dropna(subset=["transliteration", "translation"]).reset_index(drop=True)

# Split by document so sentences from the same document stay in one split
gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
train_idx, val_idx = next(gss.split(df, groups=df["oare_id"]))

df["split"] = "train"
df.loc[val_idx, "split"] = "val"

print(f"Всего записей: {len(df)}")
print(df["split"].value_counts())
df.head()

Всего записей: 1561
split
train    1404
val       157
Name: count, dtype: int64


,oare_id,transliteration,translation,split
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ...",train
1,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...,train
2,0073f2c0-524c-4bbf-915a-8c1772a4fb98,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,<gap> he did not give you a textile. He return...,train
3,009fb838-8038-42bc-ad34-5f795b3840ee,KIŠIB šu-{d}EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū...",train
4,00aa1c55-c80c-4346-a159-73ad43ab0ff7,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...,train


## 5. Train/Validation Split and Ensemble Subsets

The validation split is fixed by the shared pipeline and should remain identical across team members.

Because the dataset is currently small, each ensemble model uses a different 85% random subset of the training data. This provides model diversity without reducing each training set too aggressively.

When the dataset becomes substantially larger, consider switching to non-overlapping k-fold splits.


In [5]:
N_MODELS = 2
BOOTSTRAP_FRACTION = 0.85  # Fraction of train data used by each model

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)

print(f"train: {len(train_df)}, val: {len(val_df)}")


def make_bootstrap_splits(train_df, n_models, fraction, seed):
    """Train each model on a different random subset of the training data."""
    splits = []
    rng = np.random.default_rng(seed)
    n = len(train_df)
    for i in range(n_models):
        idx = rng.choice(n, size=int(n * fraction), replace=False)
        splits.append(train_df.iloc[idx].reset_index(drop=True))
    return splits


train_splits = make_bootstrap_splits(train_df, N_MODELS, BOOTSTRAP_FRACTION, SEED)
for i, s in enumerate(train_splits):
    print(f"Модель {i + 1}: train size = {len(s)}")

train: 1404, val: 157
Модель 1: train size = 1193
Модель 2: train size = 1193


## 6. Tokenizer and Preprocessing

ByT5 operates at the byte level, so it handles the special characters used in cuneiform transliteration without a custom vocabulary.

For the current dataset size, `google/byt5-small` is used to reduce overfitting and GPU memory usage. Consider `byt5-base` after the dataset grows.


In [6]:
MODEL_NAME = "google/byt5-small"   # Switch to "google/byt5-base" when the dataset grows
MAX_SOURCE_LEN = 256
MAX_TARGET_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def preprocess(examples):
    model_inputs = tokenizer(
        examples["transliteration"],
        max_length=MAX_SOURCE_LEN,
        truncation=True,
    )
    labels = tokenizer(
        text_target=examples["translation"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.59k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

## 7. Metrics (BLEU, chrF++)


## 8. Fine-Tuning

Each model is trained, evaluated, saved locally, and uploaded to a private Hugging Face repository. After saving, the model and trainer are removed from memory and the CUDA cache is cleared.

Only the repository name and local path are returned, so models can be loaded one at a time during inference and ensembling.


In [7]:
sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")
geom_mean = evaluate.load("chrf")



def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    refs = [[l] for l in decoded_labels]
    bleu_result = sacrebleu.compute(predictions=decoded_preds, references=refs)
    chrf_result = chrf.compute(predictions=decoded_preds, references=refs, word_order=2)  # chrF++
    geom_mean_result = np.sqrt(bleu_result["score"]*chrf_result["score"])

    return {
        "bleu": bleu_result["score"],
        "chrfpp": chrf_result["score"],
        "geom_mean": geom_mean_result
    }

In [8]:
def train_one_model(model_idx, train_subset_df, val_df, config):
    run_name = f"byt5_model_{model_idx + 1}"

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "learning_rate": config["learning_rate"],
            "batch_size": config["batch_size"],
            "max_length": MAX_SOURCE_LEN,
            "epochs": config["epochs"],
            "lora_r": None,       # Not used for ByT5 full fine-tuning; kept for consistency with the LoRA setup
            "lora_alpha": None,
            "model_idx": model_idx + 1,
            "base_model": MODEL_NAME,
        })
        mlflow.set_tag("model_architecture", MODEL_ARCHITECTURE_TAG)
        mlflow.set_tag("author", AUTHOR_NAME)  # Code runner

        model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

        train_ds = Dataset.from_pandas(train_subset_df[["transliteration", "translation"]])
        eval_ds = Dataset.from_pandas(val_df[["transliteration", "translation"]])

        train_ds = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
        eval_ds = eval_ds.map(preprocess, batched=True, remove_columns=eval_ds.column_names)

        data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

        training_args = Seq2SeqTrainingArguments(
            output_dir=f"{OUTPUT_DIR}/model_{model_idx + 1}",
            per_device_train_batch_size=config["batch_size"],
            per_device_eval_batch_size=config["batch_size"],
            learning_rate=config["learning_rate"],
            num_train_epochs=config["epochs"],
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=2,
            predict_with_generate=True,
            generation_max_length=MAX_TARGET_LEN,
            generation_num_beams=config.get("num_beams_eval", 4),
            load_best_model_at_end=True,
            metric_for_best_model="geom_mean",
            greater_is_better=True,
            logging_steps=10,
            seed=SEED + model_idx,
            report_to=[],  # Logged to MLflow manually below
            fp16=torch.cuda.is_available(),
        )

        # Newer Transformers versions use `processing_class` instead of `tokenizer`
        import inspect
        trainer_kwargs = dict(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=eval_ds,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        )
        if "processing_class" in inspect.signature(Seq2SeqTrainer.__init__).parameters:
            trainer_kwargs["processing_class"] = tokenizer
        else:
            trainer_kwargs["tokenizer"] = tokenizer

        trainer = Seq2SeqTrainer(**trainer_kwargs)

        train_result = trainer.train()
        eval_metrics = trainer.evaluate()

        mlflow.log_metrics({
            "train_loss": train_result.training_loss,
            "val_loss": eval_metrics.get("eval_loss", 0.0),
            "val_bleu": eval_metrics.get("eval_bleu", 0.0),
            "val_chrfpp": eval_metrics.get("eval_chrfpp", 0.0),
            "val_geom_mean": eval_metrics.get("val_geom_mean", 0.0),
        })

        # --- Save weights locally for ensembling without downloading again ---
        local_path = f"{OUTPUT_DIR}/final_model_{model_idx + 1}"
        trainer.save_model(local_path)
        tokenizer.save_pretrained(local_path)

        # --- Upload weights to a private Hugging Face Hub repository ---
        repo_name = f"{HF_USERNAME}/akkadian-byt5-model{model_idx + 1}"
        model.push_to_hub(repo_name, private=True)
        tokenizer.push_to_hub(repo_name, private=True)
        mlflow.set_tag("hf_repo", f"https://huggingface.co/{repo_name}")
        mlflow.set_tag("local_path", local_path)

        chrf_score = eval_metrics.get("eval_chrfpp", 0)
        bleu_score = eval_metrics.get("eval_bleu", 0)
        geom_mean_score =  eval_metrics.get("val_geom_mean_scor", 0)
        print(f"[Модель {model_idx + 1}] val_chrfpp={chrf_score:.2f}  val_BLEU={bleu_score:.2f}  "
              f"-> {repo_name} (локально: {local_path})")

        # --- Free GPU memory before training the next model ---
        del trainer
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return repo_name, local_path

## 9. Train 2–3 Models

Different hyperparameters and training subsets provide the diversity needed for the ensemble.

Adjust `MODEL_CONFIGS` to match your available compute budget.

Models are trained sequentially to avoid accumulating multiple models in GPU memory.


In [9]:
MODEL_CONFIGS = [
    # {"learning_rate": 3e-4, "batch_size": 8, "epochs": 10, "num_beams_eval": 4},
    # {"learning_rate": 5e-4, "batch_size": 8, "epochs": 12, "num_beams_eval": 4},
    {"learning_rate": 2e-4, "batch_size": 8, "epochs": 15, "num_beams_eval": 4},
    {"learning_rate": 2e-4, "batch_size": 8, "epochs": 20, "num_beams_eval": 4}
][:N_MODELS]

trained_model_paths = []  # [(repo_name, local_path), ...] — сами модели уже выгружены из памяти
for i in range(N_MODELS):
    repo_name, local_path = train_one_model(i, train_splits[i], val_df, MODEL_CONFIGS[i])
    trained_model_paths.append((repo_name, local_path))

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.20GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


[Модель 2] val_chrfpp=46.81  val_BLEU=28.15  -> Eee13/akkadian-byt5-model2 (локально: outputs/final_model_2)
🏃 View run byt5_model_2 at: https://dagshub.com/witch2256/pomello_donatello.mlflow/#/experiments/2/runs/5f11931520b64a4ca36f3e600833885a
🧪 View experiment at: https://dagshub.com/witch2256/pomello_donatello.mlflow/#/experiments/2


## 10. Inference with Beam Search and Top-K Hypotheses

For each input, generate multiple beam-search candidates for `pred_top_k` and MBR ensembling.

Models are loaded one at a time, preferring the local path and falling back to Hugging Face Hub.


In [10]:
NUM_BEAMS = 8
NUM_RETURN_SEQUENCES = 5  # top-K гипотез на пример


def load_model_for_inference(repo_name, local_path):
    """Load one model from the local path or Hugging Face Hub."""
    source = local_path if os.path.isdir(local_path) else repo_name
    model = AutoModelForSeq2SeqLM.from_pretrained(source)
    model.to(device)
    model.eval()
    return model


def unload_model(model):
    """Unload the model before loading the next one."""
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def generate_predictions(model, dataset_df, num_beams=NUM_BEAMS, num_return_sequences=NUM_RETURN_SEQUENCES,
                          batch_size=16):
    texts = dataset_df["transliteration"].tolist()
    all_top1, all_topk = [], []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        inputs = tokenizer(
            batch_texts, return_tensors="pt", truncation=True,
            max_length=MAX_SOURCE_LEN, padding=True,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=MAX_TARGET_LEN,
                num_beams=num_beams,
                num_return_sequences=num_return_sequences,
                early_stopping=True,
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        # Outputs are grouped by num_return_sequences for each input
        for i in range(len(batch_texts)):
            cand = decoded[i * num_return_sequences:(i + 1) * num_return_sequences]
            all_topk.append(cand)
            all_top1.append(cand[0])

    return all_top1, all_topk

## 11. Unified Prediction File Format

Each model produces validation and test prediction files. `pred_top_k` is stored as a JSON list for compatibility with the team pipeline.


In [11]:
def save_predictions(df_subset, top1_preds, topk_preds, model_name, split, id_col="sentence_id", has_target=True):
    data = {
        "id": df_subset[id_col].tolist(),
        "source_text": df_subset["transliteration"].tolist(),
        "pred_translation": top1_preds,
        "pred_top_k": [json.dumps(k, ensure_ascii=False) for k in topk_preds],
    }
    if has_target and "translation" in df_subset.columns:
        data["target_text"] = df_subset["translation"].tolist()

    out_df = pd.DataFrame(data)
    fname = f"{OUTPUT_DIR}/preds_{split}_{model_name}.csv"
    out_df.to_csv(fname, index=False)
    print(f"Сохранено: {fname}")
    return fname

In [12]:
model_names = [f"byt5_m{i + 1}" for i in range(N_MODELS-1)]

# --- Generate validation predictions for each model ---
val_preds_per_model = {}
for (repo_name, local_path), mname in zip(trained_model_paths, model_names):
    model = load_model_for_inference(repo_name, local_path)
    top1, topk = generate_predictions(model, val_df)
    val_preds_per_model[mname] = {"top1": top1, "topk": topk}
    save_predictions(val_df, top1, topk, mname, "val", id_col="oare_id", has_target=True)
    unload_model(model)

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Сохранено: outputs/preds_val_byt5_m1.csv


## 13. MBR Ensemble (Minimum Bayes Risk)

For each example, combine the top-K hypotheses from all models and select the candidate with the highest average pairwise utility.


In [14]:
def chrf_utility(hyp, ref):
    result = chrf.compute(predictions=[hyp], references=[[ref]], word_order=2)
    return result["score"]


def mbr_select(candidates, utility_fn):
    """Return the candidate with the highest average pairwise similarity."""
    best_hyp, best_score = candidates[0], -1.0
    for hyp in candidates:
        others = [c for c in candidates if c != hyp]
        avg_score = np.mean([utility_fn(hyp, o) for o in others]) if others else 0.0
        if avg_score > best_score:
            best_score, best_hyp = avg_score, hyp
    return best_hyp, best_score


def ensemble_mbr(preds_per_model, n_examples):
    ensembled = []
    for idx in range(n_examples):
        pooled = []
        for mname in preds_per_model:
            pooled.extend(preds_per_model[mname]["topk"][idx])
        best_hyp, _ = mbr_select(pooled, chrf_utility)
        ensembled.append(best_hyp)
    return ensembled

In [ ]:
# --- Validation ensemble and metrics ---
ensembled_val = ensemble_mbr(val_preds_per_model, len(val_df))

ensemble_val_df = pd.DataFrame({
    "id": val_df["oare_id"],
    "source_text": val_df["transliteration"],
    "target_text": val_df["translation"],
    "pred_translation": ensembled_val,
})
ensemble_val_df.to_csv(f"{OUTPUT_DIR}/preds_val_ensemble.csv", index=False)

refs = [[r] for r in val_df["translation"]]
bleu_ens = sacrebleu.compute(predictions=ensembled_val, references=refs)
chrf_ens = chrf.compute(predictions=ensembled_val, references=refs, word_order=2)
geom_mean_ens = np.sqrt(bleu_ens['score']*chrf_ens['score'])

print(f"Ансамбль (val) — BLEU: {bleu_ens['score']:.2f}, chrfpp: {chrf_ens['score']:.2f}, geom_mean: {geom_mean_ens:.2f}")

# Compare with each individual model
for mname in model_names:
    top1 = val_preds_per_model[mname]["top1"]
    b = sacrebleu.compute(predictions=top1, references=refs)
    c = chrf.compute(predictions=top1, references=refs, word_order=2)
    f = np.sqrt(b['score']*c['score'])
    print(f"{mname} (val, single-best beam) — BLEU: {b['score']:.2f}, chrfpp: {c['score']:.2f}, geom_mean: {f:.2f}")

In [ ]:
# --- Test ensemble (final submission) ---
ensembled_test = ensemble_mbr(test_preds_per_model, len(test_df))

ensemble_test_df = pd.DataFrame({
    "id": test_df["id"],
    "source_text": test_df["transliteration"],
    "pred_translation": ensembled_test,
})
ensemble_test_df.to_csv(f"{OUTPUT_DIR}/preds_test_ensemble.csv", index=False)
print(f"Сохранено: {OUTPUT_DIR}/preds_test_ensemble.csv")

## 14. Log Ensemble Results to MLflow


In [ ]:
with mlflow.start_run(run_name="byt5_ensemble_mbr2"):
    mlflow.log_params({
        "n_models": N_MODELS,
        "num_beams": NUM_BEAMS,
        "num_return_sequences": NUM_RETURN_SEQUENCES,
        "ensemble_method": "MBR (chrfpp utility)",
    })
    mlflow.log_metrics({
        "val_bleu": bleu_ens["score"],
        "val_chrfpp": chrf_ens["score"],
    })
    mlflow.set_tag("model_architecture", "byt5-ensemble-mbr")
    mlflow.set_tag("author", AUTHOR_NAME)  # Code runner
    for (repo_name, local_path), mname in zip(trained_model_paths, model_names):
        mlflow.set_tag(f"hf_repo_{mname}", f"https://huggingface.co/{repo_name}")
        mlflow.set_tag(f"local_path_{mname}", local_path)

## 15. Save model to Kaggle

In [15]:
import kagglehub

# KAGGLE_USERNAME is your Kaggle username, not HF_USERNAME
KAGGLE_USERNAME = "name"   # Replace with your Kaggle username

FRAMEWORK = "pytorch"         # AutoModelForSeq2SeqLM uses PyTorch

for i in range(1, N_MODELS + 1):
    local_path = f"outputs/final_model_{i}"
    variation = f"byt5-akkadian-v{i+2}"          # Variant name within the model

    handle = f"{KAGGLE_USERNAME}/akkadian-byt5/{FRAMEWORK}/{variation}"
    #         ^user            ^model            ^framework  ^variation

    kagglehub.model_upload(
        handle=handle,
        local_model_dir=local_path,             # Correct argument name
        version_notes=f"ByT5 fine-tuned, model {i+2}",
        license_name="Apache 2.0",              # Optional
    )
    print(f"✅ Загружено: {handle}")

Uploading Model https://api.kaggle.com/models/eeee13/akkadian-byt5/pytorch/byt5-akkadian-v3 ...


Uploading: 100%|██████████| 3.02k/3.02k [00:00<00:00, 10.6kB/s]

Upload successful: outputs/final_model_1/added_tokens.json (3KB)
Starting upload for file outputs/final_model_1/tokenizer_config.json



Uploading: 100%|██████████| 28.3k/28.3k [00:00<00:00, 95.5kB/s]

Upload successful: outputs/final_model_1/tokenizer_config.json (28KB)
Starting upload for file outputs/final_model_1/generation_config.json


...


Uploading: 100%|██████████| 1.20G/1.20G [00:20<00:00, 57.5MB/s]

Upload successful: outputs/final_model_2/model.safetensors (1GB)
Starting upload for file outputs/final_model_2/config.json



Uploading: 100%|██████████| 848/848 [00:00<00:00, 3.15kB/s]

Upload successful: outputs/final_model_2/config.json (848B)


Your model instance has been created.
Files are being processed...
See at: https://api.kaggle.com/models/eeee13/akkadian-byt5/pytorch/byt5-akkadian-v4
✅ Загружено: eeee13/akkadian-byt5/pytorch/byt5-akkadian-v4
